# 08: Fundamentos Visuais do PCA

Gráficos conceituais: variância, covariância, autovetores, transformações e workflow.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Arc
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = '#f8f8f2'
plt.rcParams['font.size'] = 11

## 1. Variância Visual

In [ ]:
np.random.seed(42)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
x_low = np.random.normal(5, 0.5, 200)
axes[0].hist(x_low, bins=25, color='#8be9fd', edgecolor='white', alpha=0.8)
axes[0].axvline(x_low.mean(), color='#ff5555', ls='--', lw=2, label=f'Média = {x_low.mean():.1f}')
axes[0].set_title(f'Baixa Variância (σ=0.5) → Var = {x_low.var():.2f}', fontweight='bold')
axes[0].legend()
x_high = np.random.normal(5, 2.0, 200)
axes[1].hist(x_high, bins=25, color='#ff79c6', edgecolor='white', alpha=0.8)
axes[1].axvline(x_high.mean(), color='#ff5555', ls='--', lw=2, label=f'Média = {x_high.mean():.1f}')
axes[1].set_title(f'Alta Variância (σ=2.0) → Var = {x_high.var():.2f}', fontweight='bold')
axes[1].legend()
plt.suptitle('Variância: dispersão dos dados', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../graficos/variancia_visual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: variancia_visual.png')

## 2. Covariância Visual

In [ ]:
np.random.seed(42)
n = 80
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
x1 = np.random.randn(n)
y1 = 0.8*x1 + np.random.randn(n)*0.3
axes[0].scatter(x1, y1, c='#50fa7b', edgecolors='white', s=60, alpha=0.8)
axes[0].set_title(f'Cov > 0\nCov = {np.cov(x1,y1)[0,1]:.2f}', fontweight='bold')
x2, y2 = np.random.randn(n), np.random.randn(n)
axes[1].scatter(x2, y2, c='#8be9fd', edgecolors='white', s=60, alpha=0.8)
axes[1].set_title(f'Cov ≈ 0\nCov = {np.cov(x2,y2)[0,1]:.2f}', fontweight='bold')
x3 = np.random.randn(n)
y3 = -0.8*x3 + np.random.randn(n)*0.3
axes[2].scatter(x3, y3, c='#ff5555', edgecolors='white', s=60, alpha=0.8)
axes[2].set_title(f'Cov < 0\nCov = {np.cov(x3,y3)[0,1]:.2f}', fontweight='bold')
plt.suptitle('Covariância: como variáveis variam juntas', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../graficos/covariancia_visual.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: covariancia_visual.png')

## 3. Matriz de Covariância - Heatmap

In [ ]:
np.random.seed(42)
cov_true = [[2.0, 1.2, -0.8], [1.2, 1.5, 0.3], [-0.8, 0.3, 1.0]]
data = np.random.multivariate_normal([0,0,0], cov_true, 200)
cov_matrix = np.cov(data.T)
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cov_matrix, cmap='RdBu_r', vmin=-3, vmax=3)
for i in range(3):
    for j in range(3):
        color = 'white' if abs(cov_matrix[i,j]) > 1.5 else 'black'
        ax.text(j, i, f'{cov_matrix[i,j]:.2f}', ha='center', va='center', fontsize=14, fontweight='bold', color=color)
ax.set_xticks(range(3)); ax.set_yticks(range(3))
ax.set_xticklabels(['X1','X2','X3'], fontsize=13); ax.set_yticklabels(['X1','X2','X3'], fontsize=13)
ax.set_title('Matriz de Covariância', fontsize=14, fontweight='bold')
plt.colorbar(im, ax=ax, shrink=0.8, label='Covariância')
plt.tight_layout()
plt.savefig('../graficos/matriz_covariancia_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: matriz_covariancia_heatmap.png')

## 4. Transformação de Vetor

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
v = np.array([1.0, 0.5])
theta = np.radians(40)
M = 1.8 * np.array([[np.cos(theta), -np.sin(theta)], [np.sin(theta), np.cos(theta)]])
v_prime = M @ v
ax.plot(0, 0, 'ko', ms=6)
ax.annotate('', xy=v, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='#8be9fd', lw=3))
ax.text(v[0]*0.5-0.15, v[1]*0.5+0.15, r'$\vec{v}$', fontsize=18, color='#8be9fd', fontweight='bold')
ax.annotate('', xy=v_prime, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='#ff79c6', lw=3))
ax.text(v_prime[0]*0.5+0.1, v_prime[1]*0.5-0.2, r"$M \cdot \vec{v}$", fontsize=16, color='#ff79c6', fontweight='bold')
arc = Arc((0,0), 0.8, 0.8, angle=0, theta1=np.degrees(np.arctan2(v[1],v[0])), theta2=np.degrees(np.arctan2(v_prime[1],v_prime[0])), color='#f1fa8c', lw=2)
ax.add_patch(arc)
ax.text(0.35, 0.25, 'rotação', fontsize=11, color='#f1fa8c', style='italic')
ax.text(0.02, 0.98, f'|v| = {np.linalg.norm(v):.2f}\n|Mv| = {np.linalg.norm(v_prime):.2f}', transform=ax.transAxes, fontsize=12, va='top', bbox=dict(boxstyle='round', facecolor='#44475a', alpha=0.9), color='#f8f8f2')
ax.set_xlim(-0.5, 3.5); ax.set_ylim(-0.5, 3.5)
ax.axhline(0, color='gray', lw=0.5, alpha=0.5); ax.axvline(0, color='gray', lw=0.5, alpha=0.5)
ax.set_aspect('equal'); ax.set_title('Transformação Linear', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../graficos/transformacao_vetor.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: transformacao_vetor.png')

## 5. Autovetores Escalam (não rotacionam)

In [ ]:
A = np.array([[3,1],[1,3]])
eigenvalues, eigenvectors = np.linalg.eig(A)
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for idx, (ax, ev, lam) in enumerate(zip(axes, eigenvectors.T, eigenvalues)):
    ax.plot(0, 0, 'ko', ms=5)
    ax.annotate('', xy=ev, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='#8be9fd', lw=3))
    ax.text(ev[0]*0.55-0.15, ev[1]*0.55+0.1, r'$\vec{v}$', fontsize=18, color='#8be9fd', fontweight='bold')
    Av = lam * ev
    ax.annotate('', xy=Av, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='#ff79c6', lw=3))
    ax.text(Av[0]*0.55+0.1, Av[1]*0.55, f'$A\\vec{{v}} = {lam:.2f}\\vec{{v}}$', fontsize=14, color='#ff79c6', fontweight='bold')
    t = np.linspace(-1, 1, 100)
    ax.plot(ev[0]*t*4, ev[1]*t*4, '--', color='gray', alpha=0.4, lw=1)
    ax.set_xlim(-4, 5); ax.set_ylim(-3, 4)
    ax.axhline(0, color='gray', lw=0.5, alpha=0.5); ax.axvline(0, color='gray', lw=0.5, alpha=0.5)
    ax.set_aspect('equal'); ax.set_title(f'Autovetor {idx+1}: λ = {lam:.2f}', fontsize=13, fontweight='bold')
plt.suptitle('Autovetores: só escalam, não rotacionam', fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('../graficos/autovetores_escalam.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: autovetores_escalam.png')

## 6. Autovetores Ortonormais

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
v1 = np.array([1, 0]); v2 = np.array([0, 1])
ax.plot(0, 0, 'ko', ms=6)
ax.annotate('', xy=v1, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='#8be9fd', lw=4))
ax.text(v1[0]+0.08, v1[1]+0.08, r'$\vec{v_1}$', fontsize=20, color='#8be9fd', fontweight='bold')
ax.annotate('', xy=v2, xytext=(0,0), arrowprops=dict(arrowstyle='->', color='#ff79c6', lw=4))
ax.text(v2[0]+0.08, v2[1]+0.08, r'$\vec{v_2}$', fontsize=20, color='#ff79c6', fontweight='bold')
arc = Arc((0,0), 0.4, 0.4, angle=0, theta1=0, theta2=90, color='#f1fa8c', lw=2.5)
ax.add_patch(arc)
ax.text(0.15, 0.15, '90°', fontsize=14, color='#f1fa8c', fontweight='bold')
ax.text(0.98, 0.02, f'||v1|| = {np.linalg.norm(v1):.0f}\n||v2|| = {np.linalg.norm(v2):.0f}', transform=ax.transAxes, fontsize=13, ha='right', bbox=dict(boxstyle='round', facecolor='#44475a', alpha=0.9), color='#f8f8f2')
ax.text(0.5, 0.02, r'$\vec{v_1} \cdot \vec{v_2} = 0$', transform=ax.transAxes, fontsize=13, ha='center', bbox=dict(boxstyle='round', facecolor='#50fa7b', alpha=0.3), color='#282a36')
ax.set_xlim(-0.3, 1.5); ax.set_ylim(-0.3, 1.5)
ax.axhline(0, color='gray', lw=0.5, alpha=0.5); ax.axvline(0, color='gray', lw=0.5, alpha=0.5)
ax.set_aspect('equal'); ax.set_title('Autovetores Ortonormais', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('../graficos/autovetores_ortonormais.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: autovetores_ortonormais.png')

## 7. Centralização dos Dados

In [ ]:
np.random.seed(42)
X = np.random.multivariate_normal([3, 5], [[1.5, 1.0], [1.0, 1.2]], 80)
X_c = X - X.mean(axis=0)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X[:,0], X[:,1], c='#bd93f9', edgecolors='white', s=60, alpha=0.7)
axes[0].scatter(*X.mean(axis=0), c='red', s=200, marker='X', zorder=5, label=f'Média ({X.mean(0)[0]:.1f}, {X.mean(0)[1]:.1f})')
axes[0].set_title('Dados Originais', fontsize=14, fontweight='bold'); axes[0].legend(); axes[0].grid(True, alpha=0.3)
axes[1].scatter(X_c[:,0], X_c[:,1], c='#50fa7b', edgecolors='white', s=60, alpha=0.7)
axes[1].scatter(0, 0, c='red', s=200, marker='X', zorder=5, label='Origem (0, 0)')
axes[1].axhline(0, color='gray', ls='--', alpha=0.5); axes[1].axvline(0, color='gray', ls='--', alpha=0.5)
axes[1].set_title('Dados Centralizados (X - μ)', fontsize=14, fontweight='bold'); axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.suptitle('Centralização: Subtrair a Média', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../graficos/centralizacao_dados.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: centralizacao_dados.png')

## 8. Workflow PCA em 4 Passos

In [ ]:
np.random.seed(42)
X_vis = np.random.multivariate_normal([3, 5], [[2.0, 1.5], [1.5, 1.0]], 60)
mu_v = X_vis.mean(axis=0); Xc_v = X_vis - mu_v
eigvals, eigvecs = np.linalg.eigh(np.cov(Xc_v.T))
idx = eigvals.argsort()[::-1]; eigvals, eigvecs = eigvals[idx], eigvecs[:, idx]
PC_v = Xc_v @ eigvecs
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
sc = ['#bd93f9', '#8be9fd', '#ff79c6', '#50fa7b']
axes[0].scatter(X_vis[:,0], X_vis[:,1], c=sc[0], edgecolors='white', s=50, alpha=0.7)
axes[0].scatter(*mu_v, c='red', s=150, marker='X', zorder=5); axes[0].set_title('1. Dados X', fontweight='bold')
axes[1].scatter(Xc_v[:,0], Xc_v[:,1], c=sc[1], edgecolors='white', s=50, alpha=0.7)
axes[1].scatter(0, 0, c='red', s=150, marker='X', zorder=5); axes[1].axhline(0, color='gray', ls='--', alpha=0.4); axes[1].axvline(0, color='gray', ls='--', alpha=0.4)
axes[1].set_title('2. Centralizar X-μ', fontweight='bold')
axes[2].scatter(Xc_v[:,0], Xc_v[:,1], c=sc[2], edgecolors='white', s=40, alpha=0.5)
for i, color in enumerate(['#ff5555', '#f1fa8c']):
    scale = np.sqrt(eigvals[i]) * 2; v = eigvecs[:, i] * scale
    axes[2].annotate('', xy=v, xytext=(0,0), arrowprops=dict(arrowstyle='->', color=color, lw=2.5))
    axes[2].text(v[0]*1.15, v[1]*1.15, f'e{i+1} (λ={eigvals[i]:.2f})', fontsize=10, color=color, fontweight='bold')
axes[2].set_title('3. Cov + Autovalores', fontweight='bold'); axes[2].set_aspect('equal')
axes[3].scatter(PC_v[:,0], PC_v[:,1], c=sc[3], edgecolors='white', s=50, alpha=0.7)
axes[3].axhline(0, color='gray', ls='--', alpha=0.4); axes[3].axvline(0, color='gray', ls='--', alpha=0.4)
axes[3].set_title('4. Projetar PC=Xc·V', fontweight='bold')
plt.suptitle('Workflow do PCA em 4 Passos', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../graficos/pca_workflow_4passos.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: pca_workflow_4passos.png')

## 9. Transformação Direta e Inversa

In [ ]:
np.random.seed(42)
X_t = np.random.multivariate_normal([0, 0], [[3, 2.5], [2.5, 3]], 60)
mu_t = X_t.mean(axis=0); Xc_t = X_t - mu_t
eigvals_t, eigvecs_t = np.linalg.eigh(np.cov(Xc_t.T))
idx = eigvals_t.argsort()[::-1]; eigvals_t, eigvecs_t = eigvals_t[idx], eigvecs_t[:, idx]
Z = Xc_t @ eigvecs_t
Z_k = Z.copy(); Z_k[:, 1] = 0
X_hat = Z_k @ eigvecs_t.T + mu_t
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].scatter(X_t[:,0], X_t[:,1], c='#bd93f9', edgecolors='white', s=60, alpha=0.7, label='X')
axes[0].scatter(*mu_t, c='red', s=150, marker='X', zorder=5); axes[0].set_title('Dados Originais X', fontweight='bold'); axes[0].legend()
axes[1].scatter(Z[:,0], Z[:,1], c='#50fa7b', edgecolors='white', s=60, alpha=0.7, label='Z')
axes[1].axhline(0, color='gray', ls='--', alpha=0.5); axes[1].axvline(0, color='gray', ls='--', alpha=0.5)
axes[1].set_title('Espaço PCA Z', fontweight='bold'); axes[1].legend()
axes[2].scatter(X_t[:,0], X_t[:,1], c='#bd93f9', edgecolors='white', s=40, alpha=0.3, label='X')
axes[2].scatter(X_hat[:,0], X_hat[:,1], c='#ff79c6', edgecolors='white', s=60, alpha=0.8, label='X̂ (1 PC)')
for i in range(len(X_t)): axes[2].plot([X_t[i,0], X_hat[i,0]], [X_t[i,1], X_hat[i,1]], '-', color='#ff5555', alpha=0.15, lw=0.8)
axes[2].set_title('Reconstrução X̂ (1 PC)', fontweight='bold'); axes[2].legend()
var_1pc = eigvals_t[0] / eigvals_t.sum() * 100
fig.text(0.5, 0.01, f'PC1 sozinho explica {var_1pc:.1f}% da variância', ha='center', fontsize=12, color='#6272a4', style='italic')
plt.suptitle('Transformação Direta (X→Z) e Inversa (Z→X̂)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../graficos/transformacao_direta_inversa.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: transformacao_direta_inversa.png')

## 10. Variância Explicada por Componente

In [ ]:
iris = load_iris()
X_scaled = StandardScaler().fit_transform(iris.data)
pca_iris = PCA(); pca_iris.fit(X_scaled)
var_r = pca_iris.explained_variance_ratio_; cum_r = np.cumsum(var_r)
fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.bar(range(1,5), var_r*100, color=['#ff79c6','#50fa7b','#8be9fd','#bd93f9'], edgecolor='white', lw=1.5)
for i, (bar, v, c) in enumerate(zip(bars, var_r, cum_r)):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1.5, f'{v*100:.1f}%\n(cum: {c*100:.1f}%)', ha='center', fontsize=11, fontweight='bold')
ax2 = ax.twinx()
ax2.plot(range(1,5), cum_r*100, 'o-', color='#f1fa8c', lw=2.5, ms=10)
ax2.axhline(95, color='#ff5555', ls='--', alpha=0.6, label='95%')
ax2.set_ylabel('Variância Acumulada (%)'); ax2.set_ylim(0, 110); ax2.legend(loc='center right')
ax.set_xlabel('Componente Principal'); ax.set_ylabel('Variância Explicada (%)')
ax.set_title('Variância Explicada por Componente (Iris)', fontweight='bold')
ax.set_xticks(range(1,5)); ax.set_ylim(0, 110)
plt.tight_layout()
plt.savefig('../graficos/variancia_explicada_componentes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Salvo: variancia_explicada_componentes.png')